In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/FundamentofDS/processed/binary/ptbxl_labeled (binary).csv')

In [3]:
df.head(5)

,ecg_id,patient_id,age,sex,height,weight,strat_fold,filename_lr,filename_hr,scp_codes,superclasses,NORM,MI,STTC,CD,HYP,is_anomaly,binary_label
0,1,15709.0,56.0,1,NaN,63.0,3,records100/00000/00001_lr,records500/00000/00001_hr,"{'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}",['NORM'],1,0,0,0,0,0,Normal
1,2,13243.0,19.0,0,NaN,70.0,2,records100/00000/00002_lr,records500/00000/00002_hr,"{'NORM': 80.0, 'SBRAD': 0.0}",['NORM'],1,0,0,0,0,0,Normal
2,3,20372.0,37.0,1,NaN,69.0,5,records100/00000/00003_lr,records500/00000/00003_hr,"{'NORM': 100.0, 'SR': 0.0}",['NORM'],1,0,0,0,0,0,Normal
3,4,17014.0,24.0,0,NaN,82.0,3,records100/00000/00004_lr,records500/00000/00004_hr,"{'NORM': 100.0, 'SR': 0.0}",['NORM'],1,0,0,0,0,0,Normal
4,5,17448.0,19.0,1,NaN,70.0,4,records100/00000/00005_lr,records500/00000/00005_hr,"{'NORM': 100.0, 'SR': 0.0}",['NORM'],1,0,0,0,0,0,Normal


In [4]:
!pip install wfdb

In [5]:
import os
import numpy as np
import wfdb
from scipy import signal

fs = 100

def remove_spikes(sig, threshold_factor=4.5):
    cleaned_sig = sig.copy()
    med = np.median(sig, axis=0)
    mad = np.median(np.abs(sig - med), axis=0)

    threshold = threshold_factor * (mad + 1e-6)
    is_spike = np.abs(sig - med) > threshold

    med_filtered = signal.medfilt(sig, kernel_size=(5, 1))
    cleaned_sig[is_spike] = med_filtered[is_spike]
    return cleaned_sig

In [7]:
lowcut, highcut = 0.5, 40.0
nyq = 0.5 * fs
b, a = signal.butter(2, [lowcut/nyq, highcut/nyq], btype='band')
despiked_signal = remove_spikes(raw_signal)

filtered_signal = signal.filtfilt(b, a, despiked_signal, axis=0)

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
final_signal = scaler.fit_transform(filtered_signal)

In [13]:
from tqdm import tqdm

num_samples = 500
df_sub = df.head(num_samples)

metrics = {
    'Despiking': {'snr': [], 'mse': [], 'prd': []},
    'Bandpass': {'snr': [], 'mse': [], 'prd': []},
    'Combined (Despiking + Bandpass)': {'snr': [], 'mse': [], 'prd': []}
}

for idx, row in tqdm(df_sub.iterrows(), total=num_samples, desc="Evaluating"):
    rec_path = row['filename_lr']
    rec_dir = os.path.dirname(rec_path)
    rec_name = os.path.basename(rec_path)
    raw_sig, _ = wfdb.rdsamp(rec_name, pn_dir=f"ptb-xl/1.0.3/{rec_dir}")

    raw_centered = signal.detrend(raw_sig, axis=0)

    sig_despiked = remove_spikes(raw_sig)

    sig_bandpass = signal.filtfilt(b, a, raw_sig, axis=0)

    sig_combined = signal.filtfilt(b, a, sig_despiked, axis=0)

    def compute_metrics(clean_sig, ref_sig):
        noise = ref_sig - clean_sig
        p_sig = np.mean(clean_sig ** 2)
        p_noise = np.mean(noise ** 2) + 1e-10

        snr = 10 * np.log10(p_sig / p_noise)
        mse = np.mean(noise ** 2)
        prd = (np.sqrt(np.sum(noise ** 2)) / np.sqrt(np.sum(ref_sig ** 2) + 1e-10)) * 100
        return snr, mse, prd

    snr_d, mse_d, prd_d = compute_metrics(sig_despiked, raw_sig)
    snr_b, mse_b, prd_b = compute_metrics(sig_bandpass, raw_centered)
    snr_c, mse_c, prd_c = compute_metrics(sig_combined, raw_centered)

    metrics['Despiking']['snr'].append(snr_d)
    metrics['Despiking']['mse'].append(mse_d)
    metrics['Despiking']['prd'].append(prd_d)

    metrics['Bandpass']['snr'].append(snr_b)
    metrics['Bandpass']['mse'].append(mse_b)
    metrics['Bandpass']['prd'].append(prd_b)

    metrics['Combined (Despiking + Bandpass)']['snr'].append(snr_c)
    metrics['Combined (Despiking + Bandpass)']['mse'].append(mse_c)
    metrics['Combined (Despiking + Bandpass)']['prd'].append(prd_c)

results = []
for method in metrics.keys():
    results.append({
        'Filtering method': method,
        'SNR (Signal-to-noise ratio)': f"{np.mean(metrics[method]['snr']):.2f}",
        'MSE (Mean-squared error)': f"{np.mean(metrics[method]['mse']):.4f}",
        'PRD (Percent Root-mean-squared difference)': f"{np.mean(metrics[method]['prd']):.2f}%"
    })

results_df = pd.DataFrame(results)
print("\nPerformance of the preprocessing in 500 samples")
print(results_df.to_string(index=False))

Evaluating: 100%|██████████| 500/500 [02:24<00:00,  3.46it/s]


Performance of the preprocessing in 500 samples
               Filtering method SNR (Signal-to-noise ratio) MSE (Mean-squared error) PRD (Percent Root-mean-squared difference)
                      Despiking                        6.49                   0.0070                                     39.31%
                       Bandpass                        9.10                   0.0100                                     37.03%
Combined (Despiking + Bandpass)                        2.18                   0.0153                                     53.85%


In [8]:
def preprocess_ecg_signal(sig):
    despiked = remove_spikes(sig)
    filtered = signal.filtfilt(b, a, despiked, axis=0)
    return scaler.fit_transform(filtered)

In [9]:
y = df['binary_label'].map({'Normal': 0, 'Abnormal': 1}).values
X = []

print(f"Total records to process: {len(df)}")
print(f"Label distribution: Normal (0) = {np.sum(y == 0)}, Abnormal (1) = {np.sum(y == 1)}")

Total records to process: 21799
Label distribution: Normal (0) = 9097, Abnormal (1) = 12702


In [ ]:
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing ECG records"):
    rec_path = row['filename_lr']
    rec_dir = os.path.dirname(rec_path)
    rec_name = os.path.basename(rec_path)
    raw_signal, _ = wfdb.rdsamp(rec_name, pn_dir=f"ptb-xl/1.0.3/{rec_dir}")

    processed_signal = preprocess_ecg_signal(raw_signal)
    X.append(processed_signal)

X = np.array(X)

print("\nCompleted!")
print(f"X feature matrix shape: {X.shape}")
print(f"y label shape: {y.shape}")

Processing ECG records: 100%|██████████| 21799/21799 [1:37:21<00:00,  3.73it/s] 



Completed!
X feature matrix shape: (21799, 1000, 12)
y label shape: (21799,)


In [ ]:
save_dir = '/content/drive/MyDrive/FundamentofDS/processed/binary'

np.save(os.path.join(save_dir, 'X_binary (3).npy'), X)
np.save(os.path.join(save_dir, 'y_binary. (3)npy'), y)
df[['patient_id', 'age', 'sex', 'strat_fold', 'binary_label']].to_csv(
    os.path.join(save_dir, 'metadata_binary (3).csv')
)